# Titanic Survival Prediction — Model Development

**Project:** Titanic Survival Prediction  
**Author:** ml-model-scientist agent  
**Date:** 2026-04-26  

This notebook trains and evaluates three classifiers (Logistic Regression, Random Forest, Gradient Boosting) on the cleaned Titanic dataset using a 70/15/15 train/validation/test split. The best model is selected by validation accuracy and evaluated on the held-out test set.

In [ ]:
import sys
sys.path.insert(0, '/Users/chen/PycharmProjects/claude-demo/titanic-survival-prediction/02_src')

import pandas as pd
import numpy as np
import json
from IPython.display import Image
from modeling import (
    split_data, scale_features, train_all_models, select_best_model,
    evaluate_model, plot_feature_importance, plot_roc_curve,
    plot_model_comparison, plot_confusion_matrix, save_model,
    FEATURE_COLS, FIGURES_DIR, MODELS_DIR
)

PROCESSED_PATH = '/Users/chen/PycharmProjects/claude-demo/titanic-survival-prediction/00_data/processed/titanic_cleaned.parquet'
df = pd.read_parquet(PROCESSED_PATH)
print(f"Dataset: {df.shape}")
df.head()

## 1. Train / Validation / Test Split (70 / 15 / 15)

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)
X_train_sc, X_val_sc, X_test_sc, scaler = scale_features(X_train, X_val, X_test)
print(f"Features used: {FEATURE_COLS}")

## 2. Model Training

In [ ]:
results, fitted_models = train_all_models(X_train, y_train, X_val, y_val, X_train_sc, X_val_sc)
best_name = select_best_model(results)
best_model = fitted_models[best_name]
X_test_best = X_test  # Random Forest uses unscaled

## 3. Model Comparison (Validation Set)

In [ ]:
rows = []
for name, res in results.items():
    row = {'Model': name}
    row.update({k: round(v, 4) for k, v in res['val'].items()})
    rows.append(row)
comparison_df = pd.DataFrame(rows).set_index('Model')
display(comparison_df)

plot_model_comparison(results, f'{FIGURES_DIR}/model_comparison.png')
Image(f'{FIGURES_DIR}/model_comparison.png')

## 4. Test Set Evaluation (Best Model: Random Forest)

In [ ]:
test_metrics = evaluate_model(best_model, X_test_best, y_test, label='Test')
pd.DataFrame([test_metrics], index=[best_name])

In [ ]:
plot_confusion_matrix(best_model, X_test_best, y_test,
                      f'{FIGURES_DIR}/confusion_matrix.png', model_name=best_name)
Image(f'{FIGURES_DIR}/confusion_matrix.png')

In [ ]:
plot_roc_curve(best_model, X_test_best, y_test,
               f'{FIGURES_DIR}/roc_curve.png', model_name=best_name)
Image(f'{FIGURES_DIR}/roc_curve.png')

## 5. Top 5 Feature Importances

In [ ]:
fi_df = plot_feature_importance(best_model, FEATURE_COLS,
                                f'{FIGURES_DIR}/feature_importance.png',
                                model_name=best_name)
print("Top 5 Features:")
display(fi_df.head(5))
Image(f'{FIGURES_DIR}/feature_importance.png')

## 6. Save Best Model

In [ ]:
save_model(best_model, scaler, f'{MODELS_DIR}/titanic_predictor_v1')
print(f"\nBest model: {best_name}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test ROC-AUC: {test_metrics['roc_auc']:.4f}")